# Bronze → Silver · Sales.SalesReason
Generado con la skill `notebooks-medallion` desde `metadata/bronze_silver/Sales.SalesReason.json`.
**No lo edites a mano**: cambia la plantilla o la metadata y vuelve a generar.

- Lee los parquet que ha dejado pyuploader en `Files/Source/AdventureWorks2022/Sales.SalesReason/year=YY/month=MM/day=DD/`, usando `.metadata/Sales.SalesReason_filehistory.json`.
- Incremental por marca de agua (la `fechahora` del último fichero procesado, guardada en `control.dbo.log_ejecucion`).
- Deduplica con `ROW_NUMBER()` por la clave `SalesReasonID`, ordenando por `LoadedBy` descendente: se queda la versión más reciente.
- Aplica los borrados (`_sql_op` en ['D']). Si llega una carga `full`, borra en Silver lo que ya no está en origen.

In [ ]:
proyecto = "GreatAdventures"
ws_datos = "GreatAdventures"
ws_control = "GreatAdventuresControl"
lh_bronze = "bronze"
lh_silver = "silver"
lh_control = "control"
carpeta_origen = "Files/Source/AdventureWorks2022"
objeto = "Sales.SalesReason"
claves = "SalesReasonID"
esquema_destino = "Sales"
tabla_destino = "SalesReason"
operaciones_borrado = "D"
modo = "auto"              # auto: incremental desde la marca de agua | reconstruir: desde la última carga full
evolucion_esquema = True   # añadir columnas nuevas del origen a Silver
simular = False            # True: lee, cuenta y registra en el log, pero no escribe en Silver
id_orquestacion = ""       # lo rellena el orquestador

In [ ]:
# Utilidades comunes (iguales en todos los notebooks generados): rutas, log en el lakehouse de control y MERGE.
import json, uuid, datetime as dt
from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T, Window

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def ruta_lakehouse(ws, lh):
    return f"abfss://{ws}@{ONELAKE}/{lh}.Lakehouse"

RUTA_CONTROL = f"{ruta_lakehouse(ws_control, lh_control)}/Tables/dbo"
ID_EJECUCION = str(uuid.uuid4())
INICIO = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

def contexto(clave, defecto=None):
    try:
        return notebookutils.runtime.context.get(clave) or defecto
    except Exception:
        return defecto

ESQUEMA_LOG_EJECUCION = T.StructType([
    T.StructField("id_ejecucion", T.StringType()),
    T.StructField("id_orquestacion", T.StringType()),
    T.StructField("proyecto", T.StringType()),
    T.StructField("capa", T.StringType()),
    T.StructField("objeto", T.StringType()),
    T.StructField("destino", T.StringType()),
    T.StructField("tipo_carga", T.StringType()),
    T.StructField("estado", T.StringType()),
    T.StructField("simulado", T.BooleanType()),
    T.StructField("inicio", T.TimestampType()),
    T.StructField("fin", T.TimestampType()),
    T.StructField("duracion_s", T.DoubleType()),
    T.StructField("ficheros_leidos", T.LongType()),
    T.StructField("filas_leidas", T.LongType()),
    T.StructField("filas_unicas", T.LongType()),
    T.StructField("filas_duplicadas", T.LongType()),
    T.StructField("filas_borrado_origen", T.LongType()),
    T.StructField("filas_insertadas", T.LongType()),
    T.StructField("filas_actualizadas", T.LongType()),
    T.StructField("filas_borradas", T.LongType()),
    T.StructField("marca_desde", T.StringType()),
    T.StructField("marca_hasta", T.StringType()),
    T.StructField("error", T.StringType()),
    T.StructField("notebook", T.StringType()),
    T.StructField("workspace", T.StringType()),
    T.StructField("usuario", T.StringType()),
    T.StructField("spark_app_id", T.StringType()),
    T.StructField("parametros", T.StringType()),
])

def registrar_ejecucion(**valores):
    """Añade una fila a control.dbo.log_ejecucion. Solo append: no hay conflictos entre notebooks en paralelo."""
    fin = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    base = dict(id_ejecucion=ID_EJECUCION, proyecto=proyecto, inicio=INICIO, fin=fin,
                duracion_s=(fin - INICIO).total_seconds(), notebook=contexto("currentNotebookName"),
                workspace=contexto("currentWorkspaceName"), usuario=contexto("userName"),
                spark_app_id=spark.sparkContext.applicationId)
    base.update(valores)
    fila = tuple(base.get(c.name) for c in ESQUEMA_LOG_EJECUCION)
    (spark.createDataFrame([fila], ESQUEMA_LOG_EJECUCION)
         .write.format("delta").mode("append").option("mergeSchema", "true")
         .save(f"{RUTA_CONTROL}/log_ejecucion"))

def es_tabla_delta(ruta):
    try:
        return DeltaTable.isDeltaTable(spark, ruta)
    except Exception:
        return False

def con_hash(df, excluir):
    """Añade _hash_fila: sha2 de las columnas de negocio. Sirve para no reescribir filas que no han cambiado."""
    cols = [c for c in df.columns if c not in excluir and not c.startswith("_")]
    return df.withColumn("_hash_fila", F.sha2(F.concat_ws("\u0001", *[F.coalesce(F.col(c).cast("string"), F.lit("\u0000")) for c in sorted(cols)]), 256))

def anadir_columnas_nuevas(ruta, df, evolucion):
    """Evolución de esquema controlada: añade al destino las columnas nuevas del origen o falla si no está permitido."""
    destino = spark.read.format("delta").load(ruta)
    nuevas = [f for f in df.schema.fields if f.name not in destino.columns]
    if not nuevas:
        return []
    if not evolucion:
        raise ValueError(f"Columnas nuevas en origen y evolucion_esquema=False: {[f.name for f in nuevas]}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in nuevas)
    spark.sql(f"ALTER TABLE delta.`{ruta}` ADD COLUMNS ({cols})")
    return [f.name for f in nuevas]

def columnas_con_tipo_cambiado(ruta, df):
    """Columnas que existen en destino con un tipo distinto al del origen."""
    actual = {f.name: f.dataType for f in spark.read.format("delta").load(ruta).schema.fields}
    return [f"{f.name}: {actual[f.name].simpleString()} -> {f.dataType.simpleString()}"
            for f in df.schema.fields if f.name in actual and actual[f.name] != f.dataType]

def fusionar(ruta, df, claves, borrar_ausentes, col_borrado=None, cambio_tipo="ignorar"):
    """MERGE contra Delta. df ya viene deduplicado por claves y con _hash_fila.
    - col_borrado (opcional): columna booleana del origen; si es True, la fila se borra en destino.
    - borrar_ausentes: borra del destino lo que no viene en el origen (instantánea completa).
    - cambio_tipo: qué hacer si una columna existente cambia de tipo en el origen.
        "ignorar"    el MERGE convierte al tipo del destino (comportamiento histórico).
        "reescribir" reescribe la tabla con el esquema del origen (solo tiene sentido con borrar_ausentes).
        "fallar"     lanza un error.
    Devuelve (insertadas, actualizadas, borradas)."""
    cols = [c for c in df.columns if c != col_borrado]
    if not es_tabla_delta(ruta):
        datos = df.filter(~F.col(col_borrado)) if col_borrado else df
        datos.select(*cols).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta)
        return spark.read.format("delta").load(ruta).count(), 0, 0
    if cambio_tipo != "ignorar":
        cambiados = columnas_con_tipo_cambiado(ruta, df.select(*cols))
        if cambiados and cambio_tipo == "fallar":
            raise ValueError(f"Cambio de tipo en {cambiados} y la tabla no se puede reescribir (borrar_ausentes=False)")
        if cambiados:
            print(f"Cambio de tipo en {cambiados}: se reescribe la tabla con el esquema de la consulta")
            antes = spark.read.format("delta").load(ruta).count()
            datos = df.filter(~F.col(col_borrado)) if col_borrado else df
            datos.select(*cols).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta)
            return spark.read.format("delta").load(ruta).count(), 0, antes
    anadir_columnas_nuevas(ruta, df.select(*cols), evolucion_esquema)
    cond = " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in claves)
    asignar = {f"`{c}`": f"s.`{c}`" for c in cols}
    m = DeltaTable.forPath(spark, ruta).alias("t").merge(df.alias("s"), cond)
    if col_borrado:
        m = m.whenMatchedDelete(condition=f"s.`{col_borrado}`")
        m = m.whenMatchedUpdate(condition=f"NOT s.`{col_borrado}` AND NOT (t._hash_fila <=> s._hash_fila)", set=asignar)
        m = m.whenNotMatchedInsert(condition=f"NOT s.`{col_borrado}`", values=asignar)
    else:
        m = m.whenMatchedUpdate(condition="NOT (t._hash_fila <=> s._hash_fila)", set=asignar)
        m = m.whenNotMatchedInsert(values=asignar)
    if borrar_ausentes:
        m = m.whenNotMatchedBySourceDelete()
    m.execute()
    met = DeltaTable.forPath(spark, ruta).history(1).select("operationMetrics").first()[0] or {}
    return (int(met.get("numTargetRowsInserted", 0)), int(met.get("numTargetRowsUpdated", 0)),
            int(met.get("numTargetRowsDeleted", 0)))

In [ ]:
CAPA = "bronze_silver"
RUTA_BRONZE = f"{ruta_lakehouse(ws_datos, lh_bronze)}/{carpeta_origen}"
RUTA_FILES = f"{ruta_lakehouse(ws_datos, lh_bronze)}/Files"
RUTA_SILVER = f"{ruta_lakehouse(ws_datos, lh_silver)}/Tables/{esquema_destino}/{tabla_destino}"
TECNICAS = {"LoadedBy", "_sql_op"}

def leer_json(ruta):
    return json.loads(notebookutils.fs.head(ruta, 50_000_000))

def marca_de_agua():
    ruta = f"{RUTA_CONTROL}/log_ejecucion"
    if not es_tabla_delta(ruta):
        return None
    return (spark.read.format("delta").load(ruta)
            .filter((F.col("capa") == CAPA) & (F.col("objeto") == objeto) & (~F.col("simulado"))
                    & F.col("estado").isin("OK", "SIN_CAMBIOS"))
            .agg(F.max("marca_hasta")).first()[0])

def lote(nombre_fichero):
    # <Tabla>_<guid de la ejecución de pyuploader>_<nnnnn>.parquet
    return nombre_fichero[: -len(".parquet")].rsplit("_", 2)[-2]

def ficheros_a_procesar(historial, marca, reconstruir):
    """Devuelve (ficheros, tipo_carga). Una carga full puede ocupar varios ficheros del mismo lote."""
    completos = [i for i, h in enumerate(historial) if h.get("mode") == "full"]
    if reconstruir:
        if not completos:
            return historial, "reconstruccion"
        lt = lote(historial[completos[-1]]["nombredelficheroparquet"])
        desde = min(i for i, h in enumerate(historial) if lote(h["nombredelficheroparquet"]) == lt)
        return historial[desde:], "reconstruccion"
    pendientes = [i for i, h in enumerate(historial) if marca is None or h["fechahora"] > marca]
    if not pendientes:
        return [], "sin_cambios"
    full_pend = [i for i in pendientes if historial[i].get("mode") == "full"]
    if full_pend:
        lt = lote(historial[full_pend[-1]]["nombredelficheroparquet"])
        desde = min(i for i, h in enumerate(historial) if lote(h["nombredelficheroparquet"]) == lt)
        return historial[desde:], "full"
    return [historial[i] for i in pendientes], "incremental"

In [ ]:
m = dict(ficheros_leidos=0, filas_leidas=0, filas_unicas=0, filas_duplicadas=0, filas_borrado_origen=0,
         filas_insertadas=0, filas_actualizadas=0, filas_borradas=0)
estado, error, tipo_carga, marca, marca_hasta = "OK", None, None, None, None
try:
    meta = leer_json(f"{RUTA_BRONZE}/.metadata/{objeto}.json")
    clave_pyuploader = [c.strip() for c in meta.get("primarykey", "").split(",") if c.strip()]
    pk = [c.strip() for c in claves.split(",") if c.strip()] or clave_pyuploader
    if not pk:
        raise ValueError(f"{objeto}: sin clave primaria en la metadata ni en .metadata de pyuploader")
    if clave_pyuploader and [c.lower() for c in pk] != [c.lower() for c in clave_pyuploader]:
        raise ValueError(f"{objeto}: la clave de la metadata {pk} no coincide con la de pyuploader {clave_pyuploader}. Revisa metadata/bronze_silver.")

    historial = sorted(leer_json(f"{RUTA_BRONZE}/.metadata/{objeto}_filehistory.json"), key=lambda h: h["fechahora"])
    reconstruir = modo == "reconstruir" or not es_tabla_delta(RUTA_SILVER)
    marca = None if reconstruir else marca_de_agua()
    ficheros, tipo_carga = ficheros_a_procesar(historial, marca, reconstruir)
    marca_hasta = ficheros[-1]["fechahora"] if ficheros else marca

    if not ficheros:
        estado = "SIN_CAMBIOS"
    else:
        rutas = [f"{RUTA_FILES}/{h['ruta'].replace(chr(92), '/')}/{h['nombredelficheroparquet']}" for h in ficheros]
        fechas = spark.createDataFrame([(h["nombredelficheroparquet"], h["fechahora"]) for h in ficheros],
                                       "_fichero string, _fechahora_fichero string")
        df = (spark.read.option("mergeSchema", "true").parquet(*rutas)
                .withColumn("_fichero", F.col("_metadata.file_name"))
                .withColumn("_orden", F.monotonically_increasing_id())
                .join(F.broadcast(fechas), "_fichero", "left")).cache()
        faltan = [c for c in pk if c not in df.columns]
        if faltan:
            raise ValueError(f"{objeto}: columnas de clave que no están en el parquet: {faltan}")
        m["ficheros_leidos"] = len(rutas)
        m["filas_leidas"] = df.count()

        if "LoadedBy" not in df.columns:
            raise ValueError(f"{objeto}: falta la columna LoadedBy de pyuploader; no se puede deduplicar")
        # Deduplicación: ROW_NUMBER() por la clave, la versión más reciente según LoadedBy.
        # Desempates (misma extracción): fichero más reciente del lote y orden dentro del fichero.
        orden = [F.to_timestamp("LoadedBy").desc(), F.col("_fechahora_fichero").desc(), F.col("_orden").desc()]
        ultimas = (df.withColumn("_n", F.row_number().over(Window.partitionBy(*pk).orderBy(*orden)))
                     .filter("_n = 1").drop("_n"))
        ops = [o.strip() for o in operaciones_borrado.split(",") if o.strip()]
        borrado = F.col("_sql_op").isin(ops) if "_sql_op" in df.columns else F.lit(False)
        ultimas = ultimas.withColumn("_borrado", borrado.cast("boolean"))
        m["filas_unicas"] = ultimas.count()
        m["filas_duplicadas"] = m["filas_leidas"] - m["filas_unicas"]
        m["filas_borrado_origen"] = ultimas.filter("_borrado").count()

        salida = (con_hash(ultimas, TECNICAS)
                  .withColumn("_cargado_bronze", F.to_timestamp("LoadedBy"))
                  .withColumn("_fichero_origen", F.col("_fichero"))
                  .withColumn("_id_ejecucion", F.lit(ID_EJECUCION))
                  .withColumn("_procesado", F.current_timestamp())
                  .drop(*TECNICAS, "_fichero", "_orden", "_fechahora_fichero"))
        if not simular:
            ins, act, bor = fusionar(RUTA_SILVER, salida, pk, borrar_ausentes=tipo_carga in ("full", "reconstruccion"),
                                     col_borrado="_borrado")
            m.update(filas_insertadas=ins, filas_actualizadas=act, filas_borradas=bor)
        df.unpersist()
except Exception as e:
    estado, error = "ERROR", f"{type(e).__name__}: {e}"[:4000]
finally:
    registrar_ejecucion(id_orquestacion=id_orquestacion or None, capa=CAPA, objeto=objeto,
                        destino=f"{lh_silver}.{esquema_destino}.{tabla_destino}", tipo_carga=tipo_carga,
                        estado=estado, simulado=bool(simular), marca_desde=marca, marca_hasta=marca_hasta, error=error,
                        parametros=json.dumps(dict(modo=modo, claves=claves, simular=simular)), **m)
if error:
    raise RuntimeError(error)

In [ ]:
notebookutils.notebook.exit(json.dumps(dict(estado=estado, objeto=objeto, tipo_carga=tipo_carga, id_ejecucion=ID_EJECUCION, **m)))